In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    summarize_index_overlap,
    collist,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    TransplantationIDReference,
    split_data,
    drop_duplicate_columns,
    common_translate,
    find_redundant_cols,
    fix_redundancies,
    collapse_col,
)

### Target Population Filtering

The transplantations in the dataset were filtered to match the target population (see [](general:tpf)). We tried again to remove empty and duplicate columns.

We matched the transplant IDs if available, then the recipient and donor IDs and finally if it was the only column available, we used the recipient ID for filtering. 

In [ ]:
data = pd.read_parquet(input_data)
targetpop = pd.read_parquet(targetpop_data)

donor = collapse_col(
    data.loc[:, ["donor_et_id_et", "donor_et_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
rec = collapse_col(
    data.loc[:, ["recipient_et_id_et", "recipient_et_iqtig"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
pairs = pd.concat([donor, rec], axis=1).set_index([0, 1]).index
match_trans = data[
    (
        ~data["transplant_et_id"].isna()
        & data["transplant_et_id"].isin(targetpop["transplant_et_id"])
    )
]
match_pairs = data[
    (
        data["transplant_et_id"].isna()
        & pairs.isin(
            targetpop.loc[:, ["donor_et_id_et", "recipient_et_id_et"]]
            .set_index(["donor_et_id_et", "recipient_et_id_et"])
            .index
        )
    )
]
match_rec = data[
    (
        donor.isna()
        & data["transplant_et_id"].isna()
        & rec.isin(targetpop["recipient_et_id_et"])
    )
]
newdata = pd.concat([match_trans, match_pairs, match_rec], axis=0)

In [ ]:
display(
    Markdown(
        f"""During filtering against the target population with the transplant ID {match_trans.shape[0]}
        ({match_trans.shape[0]/(~data["transplant_et_id"].isna()).sum():.2%}) rows were kept,
        additionally based on the donor and recipient ID pairs {match_pairs.shape[0]}
        ({match_pairs.shape[0]/(data["transplant_et_id"].isna()).sum():.2%}) rows were kept and by rows which only had a recipient ID
        available {match_rec.shape[0]}
        ({match_rec.shape[0]/(donor.isna() & data["transplant_et_id"].isna()).sum():.2%}) were added.
        """
    )
)
data = newdata
del newdata

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`IQTIG` and {term}`ET` data is not already connected (see [](general:ic)). The following table lists the different types of rows, whhich occur in this file and which ID combination they use.

In [ ]:
idcols = [
    "transplant_et_id",
    "recipient_et_id_et",
    "recipient_et_iqtig",
    "donor_et_id_et",
    "donor_et_iqtig",
]
split = split_data(data, idcols)
assert len(split) == 3, "Not 3 different row types present!?"

The {term}`ET` data always gives the transplant ID as well as the donor and recipient ID, while {term}`IQTIG` never provides the transplant ID.

In the following table and venn diagramm the overlap between the {term}`ET` and {term}`IQTIG` recipient data is shown.

In [ ]:
et = split["transplant_et_id + recipient_et_id_et + donor_et_id_et"]
summarize_index_overlap(
    et.droplevel(["transplant_et_id", "donor_et_id_et"]),
    split["recipient_et_iqtig + donor_et_iqtig"].droplevel("donor_et_iqtig"),
    "ET",
    "IQTIG Donor+Recipient",
    c=split["recipient_et_iqtig"],
    c_label="IQTIG Recipient Only",
)
del split, et

### Join Process

While the {term}`ET` data is in the wide format, where each row describes a single operation, the {term}`IQTIG` provides multiple rows for some operations. To not loose any information, we treated the transplantation data to be in a long format. The columns `heart_cause_of_death` and `liver_cause_of_death` and were removed, as they only had a few values present and were not used for kidney transplants.

In [ ]:
data.drop(columns=["heart_cause_of_death", "liver_cause_of_death"], inplace=True)

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

No row filtering was necessary (see [](general:rf)).

### Unit Conversions

We applied common translations and removed unit specifier columns (see [](general:uc)). 

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

In [ ]:
todrop = data.columns[data.columns.to_series().str.contains("_unit")]
assert (data[todrop].nunique() == 1).all(), "Multiple units detected!"
data = data.drop(columns=todrop)
display(
    Markdown(
        f"No unit conversions were necessary, as all columns only used one unit. Therefore the columns {collist(todrop)} were removed."
    )
)

The waiting codes in `waiting_state` were converted to the short representations.

In [ ]:
newcol = data["waiting_state"].str.extract("([^-]{1,2}) - .+")
analysis = pd.concat([data["waiting_state"], newcol], axis=1)
analysis = (
    analysis.groupby(analysis.columns.to_list())
    .size()
    .sort_values()
    .rename("Replaced Count")
    .reset_index()
    .rename(columns={"waiting_state": "Replaced", 0: "Replaced with"})
    .set_index("Replaced")
)
data["waiting_state"] = newcol
display(analysis)

### Consolidating Columns

We consolidated columns that appear for both {term}`ET` and {term}`IQTIG` (see [](general:crc)). The column `cold_ischemia_time_heart_min` is also used for kidney transplants, therefore we rename it to `cold_ischemia_time_min_iqtig` and `cold_ischemia_time_min` to `cold_ischemia_time_min_et`.

In [ ]:
data.rename(
    columns={
        "cold_ischemia_time_heart_min": "cold_ischemia_time_min_iqtig",
        "cold_ischemia_time_min": "cold_ischemia_time_min_et",
    },
    inplace=True,
)

In [ ]:
red = find_redundant_cols(data)
red["donor_et_id_et"] = ["donor_et_id_et", "donor_et_iqtig"]
red["recipient_et_id_et"] = ["recipient_et_id_et", "recipient_et_iqtig"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `operation_date` column as the time axis.

In [ ]:
indcols = ["transplant_et_id", "donor_et_id_et", "recipient_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["operation_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
immunosuppressants = [
    "Tacrolimus (FK-506)",
    "Cyclosporin A",
    "MMF-Cellcept",
    "Corticosteroids",
    "MMF-Myfortic",
    "Plasmapheresis",
    "Everolimus",
    "Other",
    "MoAb to lymphocytes",
    "IL2 receptor MoAb",
    "Cyclophosphamide",
    "Rituximab",
    "ATG",
    "Sirolimus",
    "Azathioprine",
    "ALG",
    "OKT3 MoAb",
    "Other Antimetabolites",
    "Other Cyclosporins",
    "Total Lymphoid Irradiation ( TLI )",
    "IL3 receptor antagonist",
    "ALG or ATG",
    "Mizoribine",
    "Anti_Pan",
    "MoAb to epitopes",
    "Methotrexate",
]


class Transplantation(TransplantationIDReference):
    aborted: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Aborted Operation",
        description="Was the operation aborted?",
        isin=["yes", "no"],
    )
    acceptable_mismatch_program: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Acceptable Mismatch Program",
        description="Was the the patient in a acceptable mismatch program?",
        isin=["AM"],
    )
    admission_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Admission Date",
        description="When was the patient admissioned to the center?",
    )
    assignment_program: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Assignment Program",
        description="Through which program was the transplant assigned?",
        isin=["AM", "ESP", "ETKAS"],
    )
    assignment_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Assignment Type",
        description="How was the transplant assigned?",
        isin=["Regular", "Rescue", "Violation"],
    )
    cold_ischemia_time_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cold Ischemia Time",
        description="What was the cold ischemia time in min?",
        ge=0,
    )
    creatinine_post_operation_umol_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Creatinine After Operation",
        description="What was the Creatinine concentration in umol/l after the operation?",
    )
    date_explanation_of_failure: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Date of Failure Explanation",
        description="When was the organ failure explained?",
    )
    date_explanation_of_failure: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Date of Failure Explanation",
        description="When was the organ failure explained?",
    )
    destination: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Transplant Destination",
        description="In what region was the transplant transported?",
        isin=["Home country", "Local", "Regional", "Abroad"],
    )
    dialysis_count: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Dialysis Count",
        description="How many dialysis were performed after the operation?",
    )
    discharge_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Discharge Date",
        description="When was the patient released from the hospital?",
    )
    discharge_reason_code: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Discharge Reason",
        description="What was reason for the discharge (code)?",
    )
    failure_reason: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Organ Failure Reason",
        description="Why did the organ fail?",
        isin=[
            "'Non-Viable' Kidney (ET)",
            "Acute Rejection (Cornea)",
            "Bleeding (ET)",
            "Chronic Rejection (Cornea)",
            "Hyperacute Rejection (ET)",
            "Infection ( not graft related ) (ET)",
            "Infection (non-renal) (ET)",
            "Infection of graft (ET)",
            "New primary renal disease (ET)",
            "Non-specific (heart/lung)",
            "Other ( renal ) (ET)",
            "Other / specify (non-renal) (ET)",
            "Patient died with functioning transplant (ET)",
            "Permanent Non-Function (ET)",
            "Primary Non-Function (non-renal) (ET)",
            "Recurrence Of Original Disease (Cornea)",
            "Recurrence of original disease (non-renal) (ET)",
            "Recurrent primary renal disease (ET)",
            "Rejection (acute / chronic) (non-renal) (ET)",
            "Rejection after stopping all immunosuppressive drugs (ET)",
            "Rejection while taking immunosuppressive drugs (acute / chronic) (ET)",
            "Rejection, Acute (heart/lung)",
            "Rejection, Chronic (AGAS [heart], BOS [lung]",
            "Removal of functioning graft (ET)",
            "Technical problems (ET)",
            "Technical problems (non-renal) (ET)",
            "Thrombosis / Infarction (ET)",
            "Thrombosis / Infarction (non-renal) (ET)",
            "unknown",
            "Vascular or Ureteric operative problems (ET)",
            "Vascular problems: not operative or rejection related (ET)",
        ],
    )
    first_hour_urine_production_ml_per_hour: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Urine Production First Hour (ml/h)",
        description="How much urine was produced in the first hour after transplantation?",
        ge=0,
    )
    hospitilization_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hospitilization Date",
        description="When was the patient admitted to the hospital?",
    )
    icd10: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Operation ICD10 Code",
        description="Which code was used to describe the operation?",
    )
    immunosuppression_a: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="1st Immunosuppressant",
        description="1st provided immunosuppressant",
        isin=immunosuppressants,
    )
    immunosuppression_a_initial: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="1st Initial Immunosuppressant",
        description="1st initially provided immunosuppressant",
        isin=immunosuppressants,
    )
    immunosuppression_b: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="2nd Immunosuppressant",
        description="2nd provided immunosuppressant",
        isin=immunosuppressants,
    )
    immunosuppression_b_initial: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="2nd Initial Immunosuppressant",
        description="2nd initially provided immunosuppressant",
        isin=immunosuppressants,
    )
    immunosuppression_c: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="3rd Immunosuppressant",
        description="3rd provided immunosuppressant",
        isin=immunosuppressants,
    )
    immunosuppression_c_initial: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="3rd Initial Immunosuppressant",
        description="3rd initially provided immunosuppressant",
        isin=immunosuppressants,
    )
    immunosuppression_d: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="4th Immunosuppressant",
        description="4th provided immunosuppressant",
        isin=immunosuppressants,
    )
    immunosuppression_d_initial: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="4th Initial Immunosuppressant",
        description="4th initially provided immunosuppressant",
        isin=immunosuppressants,
    )
    implant_side: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Implant Side",
        description="On which side was the kidney implanted?",
        isin=["Left", "Right"],
    )
    kidney_cause_of_death: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Cause of Death",
        description="Why did the patient die?",
        isin=["Infektion", "kardiovaskulär", "andere", "cerebrovaskulär", "unknown"],
    )
    kidney_function: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Kidney Function",
        description="Was the kidney functional?",
        isin=["yes", "no"],
    )
    kidney_pancreas_bleeding: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Bleeding",
        description="Was there bleeding?",
        isin=["yes"],
    )
    kidney_pancreas_complications_general: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Complications?",
        description="Were there complications?",
        isin=["yes", "no"],
    )
    kidney_pancreas_complications_other: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Kidney/Pancreas Complications?",
        description="Were there complications?",
        isin=["yes"],
    )
    kidney_pancreas_donor_compatible: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pancreas/Kidney Compatible",
        description="Were Pancreas/Kidney deemed compatible?",
        isin=["yes", "no"],
    )
    kidney_pancreas_operation_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Operation Type",
        description="What type of operation was performed?",
        isin=[
            "Isolierte Nierentransplantation",
            "Kombination Niere mit anderen Organen",
            "Kombination Pankreas mit anderen Organen",
            "Pankreastransplantation nach Nierentransplantation (PAK)",
            "Simultane Pankreas-Nierentransplantation (SPK)",
        ],
    )
    kidney_rejection: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Kidney Rejection",
        description="Was the kidney rejected?",
        isin=["yes", "no"],
    )
    kidney_retransplant: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Retransplant Kidney",
        description="Was this operation a retransplant?",
        isin=["yes", "no"],
    )
    kidney_single_or_both: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Single or Double Kidney Transplantation",
        description="Were one or two kidneys transplanted?",
        isin=[
            "isolierte Nierentransplantation (1 Organ)",
            "isolierte Nierentransplantation (2 Organe)",
        ],
    )
    last_followup_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Last Follow-Up Date",
        description="When was the last follow-up for this transplantation?",
    )
    lost_to_followup: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Lost to Follow-Up",
        description="Is this patient considered Lost to Follow-Up?",
        isin=["no", "yes"],
    )
    operation_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Operation Date",
        description="When was the patient operated?",
    )
    ops_code: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="OPS Code",
        description="Which OPS codes were assigned?",
    )
    organ: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Organ Transplanted",
        description="Which kidney was transplanted?",
        isin=["Left Kidney", "Right Kidney"],
    )
    pancreas_insulin_free: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pancreas Insulin Free",
        description="Was insulin in the pancreas?",
        isin=["yes", "no"],
    )
    pancreas_removal_necessary: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pancreas Removal",
        description="Was the pancreas removed?",
        isin=["yes", "no"],
    )
    pancreatic_drainage: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Pancreatic Drainage",
        description="What kind of drainage was used?",
        isin=["intraperitonal"],
    )
    post_operation_functional: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="PostOP Functional",
        description="Was the graft at the post operation follow-up functional?",
        isin=["yes", "no"],
    )
    postop_organ_failure_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Organ Failure Date",
        description="When did the transplant fail?",
    )
    remaining_urine_production_ml_per_hour: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Urine Production Remaining Time after First Hour (ml/h)",
        description="How much urine was produced after the first hour after transplantation?",
        ge=0,
    )
    reoperation_kidney_pancreas: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Reoperation",
        description="Was a reoperation performed?",
        isin=["yes"],
    )
    remaining_urine_production_ml_per_hour: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Urine Production Remaining Time after First Hour (ml/h)",
        description="How much urine was produced after the first hour after transplantation?",
        ge=0,
    )
    vene_drainage: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Vena Drainage",
        description="What a vene drainage was performed?",
        isin=["vena cava"],
    )
    waiting_state: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Waiting State",
        description="What was the patient state on the waiting list",
        isin=["T", "I", "NT", "HU", "HI"],
    )
    warm_ischemia_time_min: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Warm Ischemia Time (min)",
        description="The warm ischemia time in min",
        ge=0,
    )

    class Config:
        title = "Transplantation"
        description = "Each row represents a performed transplantation operation. The data is based on the 'element_transplantation.csv' file. It contains data from the ET and IQTIG."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(Transplantation, data)

In [ ]:
Transplantation.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    Transplantation.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)